# Step 1: Convert .h5 to Quantized .tflite

In [1]:
import tensorflow as tf
import numpy as np

# Load trained model
model = tf.keras.models.load_model('VGG16_To_Student\VGG16Teacher_distil_T10.h5')

# Representative dataset for quantization (using test image)
def representative_dataset():
    # Use one batch of 1 test image
    # In research, mention: "Single representative sample used for quantization"
    img = np.random.randn(1, 256, 256, 3).astype(np.float32)
    yield [img]

# Conversion to TFLite with full integer quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8  # or tf.uint8
converter.inference_output_type = tf.int8  # or tf.uint8

# Convert
tflite_model = converter.convert()

# Save
with open('model_quantized.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"Model size: {len(tflite_model)} bytes")
print(f"Target: ~112KB | Achieved: {len(tflite_model)/1024:.2f}KB")

c:\Users\My Pc\Desktop\Jisan\journal\review reponse applied soft computing\Review Progress file\R1C8\Rice-Leaf-Disease-Classification-using-Response-Based-Knowledge-Distillation\tf215_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(





INFO:tensorflow:Assets written to: C:\Users\MYPC~1\AppData\Local\Temp\tmpdtxdtzyy\assets


INFO:tensorflow:Assets written to: C:\Users\MYPC~1\AppData\Local\Temp\tmpdtxdtzyy\assets
c:\Users\My Pc\Desktop\Jisan\journal\review reponse applied soft computing\Review Progress file\R1C8\Rice-Leaf-Disease-Classification-using-Response-Based-Knowledge-Distillation\tf215_env\lib\site-packages\tensorflow\lite\python\convert.py:953: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Model size: 26352 bytes
Target: ~112KB | Achieved: 25.73KB


In [1]:
import tensorflow as tf
import numpy as np
import os

# Load the Keras model
model = tf.keras.models.load_model('VGG16_To_Student\VGG16Teacher_distil_T10.h5')

# Converter setup with int8 quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# Representative dataset (100 random samples mimicking normalized images)
def representative_dataset():
    for _ in range(100):
        yield [np.random.uniform(0, 1, size=(1, 256, 256, 3)).astype(np.float32)]

converter.representative_dataset = representative_dataset
tflite_model = converter.convert()

# Save .tflite
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

# Log sizes for your paper
model_path = r'VGG16_To_Student\VGG16Teacher_distil_T10.h5'

print(f"Original .h5 size: {os.path.getsize(model_path) / 1024:.2f} KB")
print(f"Quantized .tflite size: {os.path.getsize('model.tflite') / 1024:.2f} KB")


c:\Users\My Pc\Desktop\Jisan\journal\review reponse applied soft computing\Review Progress file\R1C8\Rice-Leaf-Disease-Classification-using-Response-Based-Knowledge-Distillation\tf215_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(





INFO:tensorflow:Assets written to: C:\Users\MYPC~1\AppData\Local\Temp\tmpmvvpodu9\assets


INFO:tensorflow:Assets written to: C:\Users\MYPC~1\AppData\Local\Temp\tmpmvvpodu9\assets
c:\Users\My Pc\Desktop\Jisan\journal\review reponse applied soft computing\Review Progress file\R1C8\Rice-Leaf-Disease-Classification-using-Response-Based-Knowledge-Distillation\tf215_env\lib\site-packages\tensorflow\lite\python\convert.py:953: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Original .h5 size: 111.12 KB
Quantized .tflite size: 25.73 KB


# Step 2: Convert .tflite to C Header File (model.h)

In [2]:
# Alternative Python method for Jupyter
with open('model_quantized.tflite', 'rb') as f:
    bytes = f.read()
    
c_array = ', '.join([f'0x{byte:02x}' for byte in bytes])
header_content = f"""
#ifndef MODEL_H
#define MODEL_H

const unsigned char g_model[] = {{{c_array}}};
const unsigned int g_model_len = {len(bytes)};

#endif
"""

with open('model.h', 'w') as f:
    f.write(header_content)

In [2]:
import os

# Path to your .tflite file
tflite_path = 'model.tflite'  # Adjust if needed

# Read the binary data
with open(tflite_path, 'rb') as f:
    data = f.read()

# Generate the C header content
header_content = 'unsigned char g_model[] = {\n'
for i, byte in enumerate(data):
    header_content += f'  0x{byte:02x},'
    if (i + 1) % 12 == 0:
        header_content += '\n'
header_content = header_content.rstrip(',') + '\n};\n'
header_content += f'unsigned int g_model_len = {len(data)};\n'

# Save to model.h
with open('model.h', 'w') as f:
    f.write(header_content)

# Log sizes (for your paper)
print(f"Generated model.h from {os.path.getsize(tflite_path) / 1024:.2f} KB .tflite")

Generated model.h from 25.73 KB .tflite


In [ ]:
import cv2
import numpy as np
from PIL import Image

# Load test image from dataset
def prepare_test_image(image_path, target_size=(256, 256)):
    # Load and resize
    img = Image.open(image_path).convert('RGB')
    img = img.resize(target_size)
    
    # Convert to array and normalize for quantization
    img_array = np.array(img).astype(np.float32)
    
    # Normalize (assuming model expects [0, 1] or [-1, 1])
    # Adjust based on your model's preprocessing
    img_array = img_array / 255.0
    img_array = (img_array - 0.5) / 0.5  # Example for [-1, 1]
    
    # Quantize to int8
    scale = 1/127.5
    zero_point = -128
    img_quantized = np.clip((img_array / scale) + zero_point, -128, 127).astype(np.int8)
    
    # Flatten for C array
    img_flat = img_quantized.flatten()
    
    return img_flat, img_array

# Generate C header
def generate_image_header(img_array, var_name="test_image"):
    c_array = ', '.join([str(int(x)) for x in img_array])
    header = f"""
#ifndef TEST_IMAGE_H
#define TEST_IMAGE_H

const int8_t {var_name}[] = {{{c_array}}};
const int {var_name}_size = {len(img_array)};

#endif
"""
    with open('test_image.h', 'w') as f:
        f.write(header)
    print(f"Image saved as C array: {len(img_array)} values")

# Usage
img_quantized, img_float = prepare_test_image("test_image.jpg")
generate_image_header(img_quantized)